In [42]:
import hashlib
import sqlite3
import bcrypt
import time
from random import getrandbits, choice
from Crypto.Cipher import AES

In [7]:
s = 'bonjour '.encode('utf-8')
s2 = 'bonjour !'.encode('utf-8')

In [8]:
s.hex(), s2.hex()

('626f6e6a6f757220', '626f6e6a6f75722021')

In [9]:
hashlib.sha256(s).hexdigest(), hashlib.sha256(s2).hexdigest()

('e0770080ea0bda10fcfd714361fa7ca451fa65375a73424c7895c8f57166a10a',
 '4fdbad33e574248f84f44f3143896ca36c25b039480a7ff2e5dbabb2d6601d83')

# EXERCICE 1:

In [14]:
alphabet = "abcdefghijklmnopqrstuvwxyz"

table = {}

for a in alphabet:
    
    for b in alphabet:
        
        for c in alphabet:
            
            
            for d in alphabet:
                
                for e in alphabet:
                    
                    mot = a + b + c + d + e
                    
                    table[hashlib.sha256(mot.encode('utf-8')).hexdigest()] = mot

len(table)

11881376

In [15]:
table["9e69e7e29351ad837503c44a5971edebc9b7e6d8601c89c284b1b59bf37afa80"]

'qwert'

In [16]:
del table

#  2 Stockage de mots de passe

In [20]:
import sqlite3

# ouvrir la connexion ou créer la base
connection = sqlite3.connect("donnees.db")

# Créer un curseur
curseur = connection.cursor()

# Créer la table
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs (name TEXT, password TEXT)")

# Exercice 2 : Niveau de sécurité −∞

In [21]:
def AjoutUtilisateur() :
    
    nom = input("login : ")
    
    curseur.execute("SELECT name FROM utilisateurs WHERE name == '" + nom + "'")
    res = curseur.fetchall()
    
    if (nom,) in res :
        
        print("Utilisateur déjà inscrit")
        
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        
        if mot_de_passe == mot_de_passe_bis :
            curseur.execute("INSERT INTO utilisateurs (name, password) VALUES (?, ?)", (nom, mot_de_passe))
            connection.commit()
        else :
            print("Erreur de saisie")

In [22]:
AjoutUtilisateur()

login : alice
Entrer votre mot de passe : soleil
Entrer à nouveau votre mot de passe : soleil


In [24]:
# 1. on affiche toutes les données de la base:
curseur.execute("SELECT * FROM utilisateurs")
curseur.fetchall()

[('alice', 'soleil')]

In [25]:
# 2. verification():

def Verification() :
    
    nom = input("login : ")
    
    mot_de_passe = input("Entrer votre mot de passe : ")
    
    curseur.execute("SELECT password FROM utilisateurs WHERE name == ?", (nom,))
    res = curseur.fetchall()
    
    if res == [] :
        print("Utilisateur inconnu")
        
    elif res[0][0] == mot_de_passe :
        print("Authentification réussie")
        
    else :
        print("Mot de passe incorrect")

In [26]:
Verification()

login : alice
Entrer votre mot de passe : soleil
Authentification réussie


In [27]:
Verification()

login : mamadou
Entrer votre mot de passe : dkkdkd
Utilisateur inconnu


# Exercice 3 : Niveau de sécurité -2

In [30]:
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_sha256 (name TEXT, password TEXT)")

def AjoutUtilisateur_sha256() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_sha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            hache = hashlib.sha256(mot_de_passe.encode('utf-8')).hexdigest()
            curseur.execute("INSERT INTO utilisateurs_sha256 (name, password) VALUES (?, ?)", (nom, hache))
            connection.commit()
        else :
            print("Erreur de saisie")

def Verification_sha256() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password FROM utilisateurs_sha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if res == [] :
        print("Utilisateur inconnu")
    elif res[0][0] == hashlib.sha256(mot_de_passe.encode('utf-8')).hexdigest() :
        print("Authentification réussie")
    else :
        print("Mot de passe incorrect")

In [32]:
AjoutUtilisateur_sha256()

login : mamadou
Entrer votre mot de passe : barry
Entrer à nouveau votre mot de passe : barry


In [33]:
curseur.execute("SELECT * FROM utilisateurs_sha256")
curseur.fetchall()

[('mamadou',
  '5a9cea259640cac7ec4428c3d584606845c2cb0d6c353252c6487b2cc0be6653')]

In [34]:
Verification_sha256()

login : mamadou
Entrer votre mot de passe : barry
Authentification réussie


# Exercice 4: Niveau de sécurité -1:

In [35]:
SEL = "R3.09-SAE3.01"
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_saltedsha256 (name TEXT, password TEXT)")

def AjoutUtilisateur_saltedsha256() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_saltedsha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            hache = hashlib.sha256((mot_de_passe + SEL).encode('utf-8')).hexdigest()
            curseur.execute("INSERT INTO utilisateurs_saltedsha256 (name, password) VALUES (?, ?)", (nom, hache))
            connection.commit()
        else :
            print("Erreur de saisie")

def Verification_saltedsha256() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password FROM utilisateurs_saltedsha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if res == [] :
        print("Utilisateur inconnu")
    elif res[0][0] == hashlib.sha256((mot_de_passe + SEL).encode('utf-8')).hexdigest() :
        print("Authentification réussie")
    else :
        print("Mot de passe incorrect")

In [36]:
AjoutUtilisateur_saltedsha256()

login : mamadou
Entrer votre mot de passe : barry
Entrer à nouveau votre mot de passe : barry


In [37]:
curseur.execute("SELECT * FROM utilisateurs_saltedsha256")
curseur.fetchall()

[('mamadou',
  'ff42caf92409bcbad8375ec05ba542f952b45c395efc0ad7b2ae8060d05d8a28')]

In [39]:
Verification_saltedsha256()

login : mamadou
Entrer votre mot de passe : barry
Authentification réussie


# Exercice 5 : Niveau de sécurité 0

In [43]:
def nouveau_sel() :
    sel = hex(getrandbits(128))[2:]
    while len(sel) < 32 :
        sel = "0" + sel
    return sel

In [44]:
nouveau_sel()

'21553b95550fba1cd6f7d137b0467eaa'

In [45]:
connection_sel = sqlite3.connect("sels.db")
curseur_sel = connection_sel.cursor()
curseur_sel.execute("CREATE TABLE IF NOT EXISTS sels (name TEXT, salt TEXT)")
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_randsaltedsha256 (name TEXT, password TEXT)")

def AjoutUtilisateur_randsaltedsha256() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_randsaltedsha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            sel = nouveau_sel()
            hache = hashlib.sha256((mot_de_passe + sel).encode('utf-8')).hexdigest()
            curseur.execute("INSERT INTO utilisateurs_randsaltedsha256 (name, password) VALUES (?, ?)", (nom, hache))
            connection.commit()
            curseur_sel.execute("INSERT INTO sels (name, salt) VALUES (?, ?)", (nom, sel))
            connection_sel.commit()
        else :
            print("Erreur de saisie")

def Verification_randsaltedsha256() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password FROM utilisateurs_randsaltedsha256 WHERE name == ?", (nom,))
    res = curseur.fetchall()
    curseur_sel.execute("SELECT salt FROM sels WHERE name == ?", (nom,))
    res_sel = curseur_sel.fetchall()
    if res == [] or res_sel == [] :
        print("Utilisateur inconnu")
    elif res[0][0] == hashlib.sha256((mot_de_passe + res_sel[0][0]).encode('utf-8')).hexdigest() :
        print("Authentification réussie")
    else :
        print("Mot de passe incorrect")

In [46]:
AjoutUtilisateur_randsaltedsha256()

login : alex
Entrer votre mot de passe : alex
Entrer à nouveau votre mot de passe : alex


In [47]:
curseur.execute("SELECT * FROM utilisateurs_randsaltedsha256")
print(curseur.fetchall())
curseur_sel.execute("SELECT * FROM sels")
print(curseur_sel.fetchall())

[('alex', 'e8fd95b2be7b5358890f8bdd573e77f8e6a64942e8d1527d16d2829fcde3b5d9')]
[('alex', '54a6e3067b5f413b86589b40f597b5d2')]


In [49]:
Verification_randsaltedsha256()

login : alex
Entrer votre mot de passe : alex
Authentification réussie


# Exercice 6 : Niveau de sécurité 1

In [50]:
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_bcrypt (name TEXT, password TEXT)")

def AjoutUtilisateur_bcrypt() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_bcrypt WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            hache = bcrypt.hashpw(mot_de_passe.encode('utf-8'), bcrypt.gensalt())
            curseur.execute("INSERT INTO utilisateurs_bcrypt (name, password) VALUES (?, ?)", (nom, hache.decode('utf-8')))
            connection.commit()
        else :
            print("Erreur de saisie")

def Verification_bcrypt() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password FROM utilisateurs_bcrypt WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if res == [] :
        print("Utilisateur inconnu")
    elif bcrypt.checkpw(mot_de_passe.encode('utf-8'), res[0][0].encode('utf-8')) :
        print("Authentification réussie")
    else :
        print("Mot de passe incorrect")

In [51]:
AjoutUtilisateur_bcrypt()

login : mamadou
Entrer votre mot de passe : barry
Entrer à nouveau votre mot de passe : barry


In [52]:
curseur.execute("SELECT * FROM utilisateurs_bcrypt")
curseur.fetchall()

[('mamadou', '$2b$12$wtQX0eorqUamENN8Wwr1uudBcNUnlu/fpKDR2J0wWny5FjgY1MXo6')]

In [53]:
Verification_bcrypt()

login : mamadou
Entrer votre mot de passe : barry
Authentification réussie


# 3 COMPARAISON:

# EXERCICE 7:

In [54]:
connection = sqlite3.connect("comparaison.db")
curseur = connection.cursor()

In [57]:
# 1. generermdp(N, n, alph) :

def generermdp(N, n, alph) :
    curseur.execute("DROP TABLE IF EXISTS utilisateurs")
    
    curseur.execute("CREATE TABLE utilisateurs (name TEXT, password TEXT)")
    for i in range(N) :
        nom = "user" + str(i)
        mot_de_passe = ""
        for j in range(n) :
            mot_de_passe = mot_de_passe + choice(alph)
        curseur.execute("INSERT INTO utilisateurs (name, password) VALUES (?, ?)", (nom, mot_de_passe))
        
    connection.commit()

In [58]:
generermdp(50, 8, "abcdefghijklmnopqrstuvwxyz0123456789")
curseur.execute("SELECT * FROM utilisateurs LIMIT 5")
curseur.fetchall()

[('user0', 'aqln7vrn'),
 ('user1', 'gww2975d'),
 ('user2', '9aw1m2q5'),
 ('user3', 'ezswu2jd'),
 ('user4', 'x96i3697')]

In [60]:
# 2.

def stockage_clair() :
    curseur.execute("DROP TABLE IF EXISTS stock_clair")
    curseur.execute("CREATE TABLE stock_clair (name TEXT, password TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        curseur.execute("INSERT INTO stock_clair (name, password) VALUES (?, ?)", (nom, mot_de_passe))
    connection.commit()

def stockage_sha256() :
    curseur.execute("DROP TABLE IF EXISTS stock_sha256")
    curseur.execute("CREATE TABLE stock_sha256 (name TEXT, password TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        hache = hashlib.sha256(mot_de_passe.encode('utf-8')).hexdigest()
        curseur.execute("INSERT INTO stock_sha256 (name, password) VALUES (?, ?)", (nom, hache))
    connection.commit()

def stockage_saltedsha256() :
    curseur.execute("DROP TABLE IF EXISTS stock_saltedsha256")
    curseur.execute("CREATE TABLE stock_saltedsha256 (name TEXT, password TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        hache = hashlib.sha256((mot_de_passe + SEL).encode('utf-8')).hexdigest()
        curseur.execute("INSERT INTO stock_saltedsha256 (name, password) VALUES (?, ?)", (nom, hache))
    connection.commit()

def stockage_randsaltedsha256() :
    curseur.execute("DROP TABLE IF EXISTS stock_randsaltedsha256")
    curseur.execute("CREATE TABLE stock_randsaltedsha256 (name TEXT, password TEXT)")
    curseur_sel.execute("DROP TABLE IF EXISTS stock_sels")
    curseur_sel.execute("CREATE TABLE stock_sels (name TEXT, salt TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        sel = nouveau_sel()
        hache = hashlib.sha256((mot_de_passe + sel).encode('utf-8')).hexdigest()
        curseur.execute("INSERT INTO stock_randsaltedsha256 (name, password) VALUES (?, ?)", (nom, hache))
        curseur_sel.execute("INSERT INTO stock_sels (name, salt) VALUES (?, ?)", (nom, sel))
    connection.commit()
    connection_sel.commit()

def stockage_bcrypt() :
    curseur.execute("DROP TABLE IF EXISTS stock_bcrypt")
    curseur.execute("CREATE TABLE stock_bcrypt (name TEXT, password TEXT)")
    curseur.execute("SELECT name, password FROM utilisateurs")
    for (nom, mot_de_passe) in curseur.fetchall() :
        hache = bcrypt.hashpw(mot_de_passe.encode('utf-8'), bcrypt.gensalt())
        curseur.execute("INSERT INTO stock_bcrypt (name, password) VALUES (?, ?)", (nom, hache.decode('utf-8')))
    connection.commit()

In [77]:
stockage_clair()
stockage_sha256()
stockage_saltedsha256()
stockage_randsaltedsha256()
stockage_bcrypt()

In [67]:
# 3.

def mdp_clair(nom) :
    curseur.execute("SELECT password FROM utilisateurs WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0]

def auth_clair(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur.execute("SELECT password FROM stock_clair WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0] == mot_de_passe

def auth_sha256(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur.execute("SELECT password FROM stock_sha256 WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0] == hashlib.sha256(mot_de_passe.encode('utf-8')).hexdigest()

def auth_saltedsha256(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur.execute("SELECT password FROM stock_saltedsha256 WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0] == hashlib.sha256((mot_de_passe + SEL).encode('utf-8')).hexdigest()

def auth_randsaltedsha256(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur_sel.execute("SELECT salt FROM stock_sels WHERE name == ?", (nom,))
    sel = curseur_sel.fetchall()[0][0]
    curseur.execute("SELECT password FROM stock_randsaltedsha256 WHERE name == ?", (nom,))
    return curseur.fetchall()[0][0] == hashlib.sha256((mot_de_passe + sel).encode('utf-8')).hexdigest()

def auth_bcrypt(nom) :
    mot_de_passe = mdp_clair(nom)
    curseur.execute("SELECT password FROM stock_bcrypt WHERE name == ?", (nom,))
    return bcrypt.checkpw(mot_de_passe.encode('utf-8'), curseur.fetchall()[0][0].encode('utf-8'))

In [68]:
auth_clair("user0"), auth_sha256("user0"), auth_saltedsha256("user0"), auth_randsaltedsha256("user0"), auth_bcrypt("user0")

(True, True, True, True, True)

In [69]:
# 4.

def temps_authentification(auth) :
    curseur.execute("SELECT name FROM utilisateurs")
    noms = curseur.fetchall()
    debut = time.time()
    for (nom,) in noms :
        if not auth(nom) :
            print("Échec pour", nom)
    return time.time() - debut

In [70]:
print("clair                   :", temps_authentification(auth_clair))
print("SHA-256                 :", temps_authentification(auth_sha256))
print("SHA-256 + sel global    :", temps_authentification(auth_saltedsha256))
print("SHA-256 + sel aléatoire :", temps_authentification(auth_randsaltedsha256))
print("bcrypt                  :", temps_authentification(auth_bcrypt))

clair                   : 0.0029828548431396484
SHA-256                 : 0.0024895668029785156
SHA-256 + sel global    : 0.0024521350860595703
SHA-256 + sel aléatoire : 0.0036163330078125
bcrypt                  : 10.767648935317993


# Bonus: Niveau de sécurité 2:

In [71]:
connection = sqlite3.connect("donnees.db")
curseur = connection.cursor()

In [72]:
CLE = b"YELLOW SUBMARINE"
curseur.execute("CREATE TABLE IF NOT EXISTS utilisateurs_aes (name TEXT, password TEXT, nonce TEXT)")

def AjoutUtilisateur_aes() :
    nom = input("login : ")
    curseur.execute("SELECT name FROM utilisateurs_aes WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if (nom,) in res :
        print("Utilisateur déjà inscrit")
    else :
        mot_de_passe = input("Entrer votre mot de passe : ")
        mot_de_passe_bis = input("Entrer à nouveau votre mot de passe : ")
        if mot_de_passe == mot_de_passe_bis :
            hache = bcrypt.hashpw(mot_de_passe.encode('utf-8'), bcrypt.gensalt())
            cipher = AES.new(CLE, AES.MODE_CTR)
            chiffre = cipher.encrypt(hache)
            curseur.execute("INSERT INTO utilisateurs_aes (name, password, nonce) VALUES (?, ?, ?)",
                            (nom, chiffre.hex(), cipher.nonce.hex()))
            connection.commit()
        else :
            print("Erreur de saisie")

def Verification_aes() :
    nom = input("login : ")
    mot_de_passe = input("Entrer votre mot de passe : ")
    curseur.execute("SELECT password, nonce FROM utilisateurs_aes WHERE name == ?", (nom,))
    res = curseur.fetchall()
    if res == [] :
        print("Utilisateur inconnu")
    else :
        chiffre = bytes.fromhex(res[0][0])
        nonce = bytes.fromhex(res[0][1])
        cipher = AES.new(CLE, AES.MODE_CTR, nonce=nonce)
        hache = cipher.decrypt(chiffre)
        if bcrypt.checkpw(mot_de_passe.encode('utf-8'), hache) :
            print("Authentification réussie")
        else :
            print("Mot de passe incorrect")

In [73]:
AjoutUtilisateur_aes()


login : mamadou
Entrer votre mot de passe : barry
Entrer à nouveau votre mot de passe : barry


In [74]:
curseur.execute("SELECT * FROM utilisateurs_aes")
curseur.fetchall()

[('mamadou',
  '40bfc0263a0334beaaa1c52c38a7254a6a74922facef718db12fd747a98bf3bcfe7d8d320aa21411f5468ee2cf60ca3b85fe9854706b241570caa9c1',
  '2589d1111b57b99a')]

In [76]:
Verification_aes()

login : mamadou
Entrer votre mot de passe : barry
Authentification réussie
